# Phase 2 — SFT Results Only (Load Pre-trained Adapters)

**Purpose:** Skip training — load all 5 trial adapters from `MyDrive/sft_trials` and produce a complete `sft_results.csv`.

**Prerequisite:** All 5 trial folders (T1–T5) must exist in `MyDrive/sft_trials/`, each containing `adapter_config.json` and `adapter_model.safetensors`.

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1: Install dependencies
!pip install -q --no-cache-dir \
    numpy==1.26.4 \
    pandas==2.2.2 \
    transformers==4.52.4 \
    trl==0.9.6 \
    peft==0.12.0 \
    accelerate==1.7.0 \
    bitsandbytes==0.46.0 \
    datasets==3.5.0 \
    evaluate==0.4.3 \
    bert-score==0.3.13 \
    sacrebleu==2.4.3

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3: Imports
import gc
import glob
import json
import os

import evaluate
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: Tesla T4


In [ ]:
# Cell 4: Config — update paths here if needed
MODEL_ID         = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
PROMPTS_PATH     = "/content/drive/MyDrive/Colab Notebooks/data/test_prompts.json"
DRIVE_TRIALS_DIR = "/content/drive/MyDrive/sft_trials"   # root folder containing T1..T5
SFT_RESULTS_PATH = "/content/drive/MyDrive/sft_results.csv"
MAX_NEW_TOKENS   = 200
TEMPERATURE      = 0.7
TOP_P            = 0.9

# Trial metadata — mirrors original training configs for CSV record-keeping
TRIAL_CONFIGS = [
    {
        "name": "T1",
        "desc": "Minimal LoRA (r=4), 1 epoch",
        "lora": {"r": 4, "lora_alpha": 8, "target_modules": ["q_proj", "v_proj"]},
        "train": {"learning_rate": 2e-4, "per_device_train_batch_size": 4,
                  "gradient_accumulation_steps": 4, "num_train_epochs": 1},
    },
    {
        "name": "T2",
        "desc": "r=8, same modules, 2 epochs",
        "lora": {"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj"]},
        "train": {"learning_rate": 2e-4, "per_device_train_batch_size": 4,
                  "gradient_accumulation_steps": 4, "num_train_epochs": 2},
    },
    {
        "name": "T3",
        "desc": "r=8, all attn modules (SAFE VERSION)",
        "lora": {"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj", "o_proj"]},
        "train": {"learning_rate": 1e-4, "per_device_train_batch_size": 2,
                  "gradient_accumulation_steps": 8, "num_train_epochs": 2},
    },
    {
        "name": "T4",
        "desc": "r=16, extended LoRA (safe full-ish)",
        "lora": {"r": 16, "lora_alpha": 32,
                 "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                    "gate_proj", "up_proj", "down_proj"]},
        "train": {"learning_rate": 5e-5, "per_device_train_batch_size": 1,
                  "gradient_accumulation_steps": 16, "num_train_epochs": 1},
    },
    {
        "name": "T5",
        "desc": "r=8, high LR (5e-4), 1 epoch",
        "lora": {"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj"]},
        "train": {"learning_rate": 5e-4, "per_device_train_batch_size": 2,
                  "gradient_accumulation_steps": 8, "num_train_epochs": 1},
    },
]

print(f"Trials to evaluate: {[c['name'] for c in TRIAL_CONFIGS]}")
print(f"Adapter root: {DRIVE_TRIALS_DIR}")

Trials to evaluate: ['T1', 'T2', 'T3', 'T4', 'T5']
Adapter root: /content/drive/MyDrive/sft_trials


In [ ]:
# Cell 5: Load test prompts + sacrebleu
with open(PROMPTS_PATH) as f:
    data = json.load(f)

test_prompts    = [p["prompt"]    for p in data["prompts"]]
test_references = [p["reference"] for p in data["prompts"]]

empty = [i + 1 for i, r in enumerate(test_references) if not r.strip()]
if empty:
    raise ValueError(f"Missing references for prompt IDs: {empty}")

sacrebleu = evaluate.load("sacrebleu")
print(f"Loaded {len(test_prompts)} test prompts.")

Loaded 10 test prompts.


In [ ]:
# Cell 6: Helpers — 4-bit BnB config + evaluation function + val_loss reader

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def evaluate_model(model, tokenizer, prompts, references):
    """Run BLEU + BERTScore on test prompts. Returns (mean_bleu, mean_bert_f1, generated_texts)."""
    model.eval()
    generated = []
    for prompt in prompts:
        formatted = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        generated.append(text)

    bleu_scores = [
        sacrebleu.compute(predictions=[g], references=[[r]])["score"]
        for g, r in zip(generated, references)
    ]
    _, _, F1 = bert_score_fn(
        generated, references, lang="en", rescale_with_baseline=True, verbose=False
    )
    bert_f1 = [f.item() for f in F1]

    return (
        round(sum(bleu_scores) / len(bleu_scores), 4),
        round(sum(bert_f1) / len(bert_f1), 4),
        generated,
    )


def get_val_loss(trial_dir):
    """Read last eval_loss from trainer_state.json inside the checkpoint subfolder."""
    pattern = os.path.join(trial_dir, "checkpoint-*", "trainer_state.json")
    matches = sorted(
        glob.glob(pattern),
        key=lambda p: int(os.path.basename(os.path.dirname(p)).split("-")[1]),
    )
    if not matches:
        return None
    with open(matches[-1]) as f:
        state = json.load(f)
    eval_entries = [e for e in state.get("log_history", []) if "eval_loss" in e]
    if eval_entries:
        return round(eval_entries[-1]["eval_loss"], 4)
    return None


print("Helpers defined.")

Helpers defined.


In [ ]:
# Cell 7: Evaluation loop — loads each adapter, evaluates, frees GPU memory
results = []

for cfg in TRIAL_CONFIGS:
    trial_name   = cfg["name"]
    adapter_path = os.path.join(DRIVE_TRIALS_DIR, trial_name)

    print(f"\n{'='*70}")
    print(f"  TRIAL {trial_name}: {cfg['desc']}")
    print(f"  Adapter: {adapter_path}")
    print(f"{'='*70}")

    if not os.path.isdir(adapter_path):
        print(f"  WARNING: {adapter_path} not found — skipping.")
        continue

    # Load tokenizer from adapter folder (it was saved there during training)
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # left-pad for generation

    # Load base model + adapter
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    # Evaluate on test prompts
    print("Evaluating on test prompts...")
    mean_bleu, mean_bert_f1, generated_texts = evaluate_model(
        model, tokenizer, test_prompts, test_references
    )

    # Recover val_loss from saved checkpoint inside the trial folder
    val_loss = get_val_loss(adapter_path)

    results.append({
        "trial":          trial_name,
        "desc":           cfg["desc"],
        "r":              cfg["lora"]["r"],
        "lora_alpha":     cfg["lora"]["lora_alpha"],
        "target_modules": ", ".join(cfg["lora"]["target_modules"]),
        "lr":             cfg["train"]["learning_rate"],
        "batch_size":     cfg["train"]["per_device_train_batch_size"],
        "grad_accum":     cfg["train"]["gradient_accumulation_steps"],
        "epochs":         cfg["train"]["num_train_epochs"],
        "val_loss":       val_loss,
        "mean_bleu":      mean_bleu,
        "mean_bert_f1":   mean_bert_f1,
        "adapter_path":   adapter_path,
        "generated_texts": generated_texts,
    })

    print(
        f"\n  {trial_name} | val_loss={val_loss} | "
        f"BLEU={mean_bleu} | BERTScore F1={mean_bert_f1}"
    )

    # Free GPU memory before next trial
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nAll {len(results)} trials evaluated.")


  TRIAL T1: Minimal LoRA (r=4), 1 epoch
  Adapter: /content/drive/MyDrive/sft_trials/T1


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Evaluating on test prompts...


...


  T3 | val_loss=0.9819 | BLEU=5.2721 | BERTScore F1=0.2155

  TRIAL T4: r=16, extended LoRA (safe full-ish)
  Adapter: /content/drive/MyDrive/sft_trials/T4
Evaluating on test prompts...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



  T4 | val_loss=0.9801 | BLEU=5.1978 | BERTScore F1=0.2198

  TRIAL T5: r=8, high LR (5e-4), 1 epoch
  Adapter: /content/drive/MyDrive/sft_trials/T5
Evaluating on test prompts...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



  T5 | val_loss=0.9873 | BLEU=5.6923 | BERTScore F1=0.1872

All 5 trials evaluated.


In [ ]:
# Cell 8: Results comparison table + save sft_results.csv
cols = [
    "trial", "desc", "r", "lora_alpha", "target_modules",
    "lr", "batch_size", "grad_accum", "epochs",
    "val_loss", "mean_bleu", "mean_bert_f1",
]

df_sft = pd.DataFrame(results)[cols]
pd.set_option("display.max_colwidth", 55)
print(df_sft.to_string(index=False))

df_sft.to_csv(SFT_RESULTS_PATH, index=False)
print(f"\nSaved to {SFT_RESULTS_PATH}")

trial                                 desc  r  lora_alpha                                                target_modules      lr  batch_size  grad_accum  epochs  val_loss  mean_bleu  mean_bert_f1
   T1          Minimal LoRA (r=4), 1 epoch  4           8                                                q_proj, v_proj 0.00020           4           4       1    1.0083     4.8801        0.1889
   T2          r=8, same modules, 2 epochs  8          16                                                q_proj, v_proj 0.00020           4           4       2    0.9867     6.4503        0.2210
   T3 r=8, all attn modules (SAFE VERSION)  8          16                                        q_proj, v_proj, o_proj 0.00010           2           8       2    0.9819     5.2721        0.2155
   T4  r=16, extended LoRA (safe full-ish) 16          32 q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj 0.00005           1          16       1    0.9801     5.1978        0.2198
   T5         r=8, high L

In [ ]:
# Cell 9: Select best SFT trial
# Ranking: BERTScore F1 (primary) → BLEU (secondary) → val_loss (tiebreaker, lower is better)
df_ranked = df_sft.sort_values(
    by=["mean_bert_f1", "mean_bleu", "val_loss"],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

best     = df_ranked.iloc[0]
best_idx = df_ranked.index[0]   # index into original results list

# Map trial name back to results list
best_result = next(r for r in results if r["trial"] == best["trial"])

print("\n" + "="*60)
print(f"  BEST SFT TRIAL: {best['trial']}")
print(f"  Description:    {best['desc']}")
print(f"  BERTScore F1:   {best['mean_bert_f1']}")
print(f"  BLEU:           {best['mean_bleu']}")
print(f"  Val Loss:       {best['val_loss']}")
print(f"  Adapter path:   {best_result['adapter_path']}")
print("="*60)

BEST_SFT_ADAPTER = best_result["adapter_path"]
print(f"\nUse this path in 02_dpo_training.ipynb:\n  {BEST_SFT_ADAPTER}")


  BEST SFT TRIAL: T2
  Description:    r=8, same modules, 2 epochs
  BERTScore F1:   0.221
  BLEU:           6.4503
  Val Loss:       0.9867
  Adapter path:   /content/drive/MyDrive/sft_trials/T2

Use this path in 02_dpo_training.ipynb:
  /content/drive/MyDrive/sft_trials/T2


In [ ]:
# Cell 10: Print generated outputs for best trial (for report)
print(f"Generated outputs from best trial ({best['trial']}):")
for i, (prompt, gen, ref) in enumerate(
    zip(test_prompts, best_result["generated_texts"], test_references)
):
    print(f"\n[{i+1}] {prompt[:80]}")
    print(f"  Generated : {gen[:200]}")
    print(f"  Reference : {ref[:200]}")

Generated outputs from best trial (T2):

[1] What is the greenhouse effect and how does it contribute to climate change?
  Generated : The greenhouse effect is a process in which greenhouse gases in the atmosphere absorb the heat from the sun and keep it trapped inside the Earth's atmosphere. This keeps the Earth's surface temperatur
  Reference : The greenhouse effect is a natural process in which certain gases in Earth's atmosphere, such as carbon dioxide, methane, and water vapor, trap heat from the Sun. Sunlight enters the atmosphere and wa

[2] Explain the difference between supervised and unsupervised learning in machine l
  Generated : Supervised learning is a method of training a machine to recognize a pattern in a data set, such as the images or text it is given. This type of learning involves defining a set of training examples a
  Reference : Supervised learning is a type of machine learning where the model is trained using labeled data. This means the input data is paired w